In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [1]:
import condacolab
condacolab.check()

✨🍰✨ Everything looks OK!


# Clone or Update

In [2]:
!git clone https://github.com/magnimel/jersey-number-pipeline.git

Cloning into 'jersey-number-pipeline'...
remote: Enumerating objects: 1009, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 1009 (delta 174), reused 165 (delta 138), pack-reused 758 (from 1)
Receiving objects: 100% (1009/1009), 89.80 MiB | 15.42 MiB/s, done.
Resolving deltas: 100% (458/458), done.


In [3]:
import os
os.chdir('/content/jersey-number-pipeline/')

In [10]:
# !git switch fahd-lstm

Branch 'fahd-lstm' set up to track remote branch 'fahd-lstm' from 'origin'.
Switched to a new branch 'fahd-lstm'


In [24]:
# RUN THIS CELL TO UPDATE YOUR CODE AFTER YOU COMMIT LOCALLY
!git fetch
!git pull

remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 424 bytes | 424.00 KiB/s, done.
From https://github.com/magnimel/jersey-number-pipeline
   1573379..40dbaa1  fahd-lstm  -> origin/fahd-lstm
Updating 1573379..40dbaa1
Fast-forward
 downloadables.py | 3 +++
 1 file changed, 3 insertions(+)


# Setup

In [13]:
!pip install gdown

In [14]:
!python ./setup.py dataset

Creating conda env 'jersey' (python=3.9)...
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/jersey

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    icu-78.3                   |       h33c6efd_0        12.1 MB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.7.4             |     

In [25]:
!conda run -n jersey --no-capture-output python downloadables.py

--- Jersey-2023 splits ---
  OK (already present): train.zip
  Already extracted: train.zip
  OK (already present): test.zip
  Already extracted: test.zip
  OK (already present): challenge.zip
  Already extracted: challenge.zip

--- Model checkpoints ---
  OK (already present): parseq-bb5792a6.pt
  OK (already present): parseq_epoch=24-step=2575-val_accuracy=95.6044-val_NED=96.3255.ckpt
  OK (already present): legibility_resnet34_soccer_20240215.pth
  OK (already present): market1501_resnet50_256_128_epoch_120.ckpt
  OK (already present): dukemtmcreid_resnet50_256_128_epoch_120.ckpt

--- Extra data files ---
  OK (already present): SoccerNetLegibility.zip
  Already extracted: SoccerNetLegibility.zip
  OK (already present): soccer_lmdb.zip
  Already extracted: soccer_lmdb.zip

--- Reid model copy ---
  OK (already present): market1501_resnet50_256_128_epoch_120.ckpt
  OK (already present): dukemtmcreid_resnet50_256_128_epoch_120.ckpt

Done.


# Run Code Below

### (Optional) Subset configuration
Edit `SUBSET_FRACTION` below to run on a random slice of the data. `1.0` = full run.

In [ ]:
SUBSET_FRACTION = 0.0001
RANDOM_SEED     = 42

if SUBSET_FRACTION < 1.0:
    import os, random, json
    import configuration as cfg

    random.seed(RANDOM_SEED)
    split = 'train'
    images_root = os.path.join(
        cfg.dataset['SoccerNet']['root_dir'],
        cfg.dataset['SoccerNet'][split]['images']
    )
    all_images = []
    for tracklet in os.listdir(images_root):
        track_dir = os.path.join(images_root, tracklet)
        if not os.path.isdir(track_dir): continue
        for f in os.listdir(track_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(track_dir, f))

    n = max(1, int(len(all_images) * SUBSET_FRACTION))
    subset = random.sample(all_images, n)
    print(f'Subset: {n} / {len(all_images)} images ({SUBSET_FRACTION*100:.1f}%)')

    # Write a filtered legible JSON that main.py will pick up via --subset flag
    subset_file = os.path.join(
        cfg.dataset['SoccerNet']['working_dir'], 'subset_images.json'
    )
    with open(subset_file, 'w') as fh:
        json.dump(subset, fh)
    print(f'Subset image list -> {subset_file}')
    SUBSET_ARG = f'--subset {subset_file}'
else:
    SUBSET_ARG = ''
    print('Running on full dataset.')

Subset: 73 / 733001 images (0.0%)
Subset image list -> ./out/SoccerNetResults/subset_images.json


### Run pipeline

In [ ]:
!conda run -n jersey --no-capture-output python main.py SoccerNet test --resume --esrgan

Detected completed stages: ['soccer_ball_filter']
Generate features
Loading model from ./reid/centroids-reid//models/market1501_resnet50_256_128_epoch_120.ckpt...
using GPU
ReidTransforms...
Preparing tracklet data...
[REID] Calibrating batch size ...
  batch_size=   8  →  390.1 img/s  (trials: ['0.31s', '0.02s', '0.02s'])
  batch_size=  16  →  441.1 img/s  (trials: ['0.38s', '0.04s', '0.04s'])
  batch_size=  32  →  455.4 img/s  (trials: ['0.58s', '0.07s', '0.07s'])
  batch_size=  64  →  465.7 img/s  (trials: ['1.02s', '0.14s', '0.13s'])
  batch_size= 128  →  482.3 img/s  (trials: ['1.93s', '0.27s', '0.25s'])
  batch_size= 256  →  493.4 img/s  (trials: ['3.74s', '0.52s', '0.51s'])
[REID] Selected batch_size=256 (493.4 img/s)
Generating features in batches (batch_size=256, num_workers=2)...
100% 2206/2206 [21:33<00:00,  1.70it/s]
Saving features...
Done generating features
Identify and remove outliers
100% 1211/1211 [00:33<00:00, 36.18it/s]
Done removing outliers
Classifying Legibility:

In [ ]:
!python evaluate.py --pred ./out/SoccerNetResults/final_results.json --gt ./data/SoccerNet/test/test_gt.json

In [ ]:
!zip -r -q /content/all_run.zip /content/jersey-number-pipeline/out/

In [ ]:
files.download("/content/all_run.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!rm /content/img_archive.zip

In [ ]:
!rm -rf /content/jersey-number-pipeline/out/SoccerNetResults/*